# Taller de Feature Engineering: Seleccion y creacion de variables

**Bootcamp:** Python - PY4
**Modulo:** Machine Learning Avanzado y Feature Engineering  
**Fase 5 : AutoML**

## De que trata este material

Ya vimos como optimizar un modelo especifico con Optuna. Ahora vamos a mirar dos herramientas de **AutoML**, que automatizan buena parte del proceso de probar modelos y pipelines:

1. **PyCaret**: vamos a entender que es y como funciona, mostrando el codigo tipico como ejemplo, pero **sin ejecutarlo** en este notebook (mas abajo explicamos por que).
2. **TPOT**: vamos a instalarlo y correrlo de verdad, con codigo ejecutable. TPOT usa **programacion genetica** para evolucionar pipelines completos de Machine Learning.

Este material vale puntos extra, tal como indica la consigna del taller.


## Paso 1: Configuracion y reconstruccion rapida del trabajo previo

Igual que en el resto de la sesion 3, recargamos el dataset y reconstruimos rapidamente las features de la sesion 2 (Winsorization, Target Encoding, variables ciclicas), para tener listos `X_train_fe` y `X_test_fe`.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42

URL = "https://raw.githubusercontent.com/Camille-Le-Roy/data_analytics_resources/main/Module_02_machine_learning_and_feature_engineering/datasets/bikeshare_Washington.csv"
df = pd.read_csv(URL)

features_base = ["season", "yr", "mnth", "hr", "holiday", "weekday",
                  "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"]

X = df[features_base].copy()
y = df["cnt"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Entrenamiento:", X_train.shape, " Prueba:", X_test.shape)


Entrenamiento: (14016, 12)  Prueba: (3504, 12)


In [ ]:
def calcular_limites(serie, percentil_bajo=5, percentil_alto=95):
    limite_bajo = np.percentile(serie, percentil_bajo)
    limite_alto = np.percentile(serie, percentil_alto)
    return limite_bajo, limite_alto

def aplicar_winsorization(X, columna, limites):
    limite_bajo, limite_alto = limites
    es_outlier = ((X[columna] < limite_bajo) | (X[columna] > limite_alto)).astype(int)
    valor_winsorizado = X[columna].clip(lower=limite_bajo, upper=limite_alto)
    return valor_winsorizado, es_outlier

def transformar_ciclica(valor, periodo):
    seno = np.sin(2 * np.pi * valor / periodo)
    coseno = np.cos(2 * np.pi * valor / periodo)
    return seno, coseno

# Winsorization
limites_windspeed = calcular_limites(X_train["windspeed"])
limites_hum = calcular_limites(X_train["hum"])

X_train["windspeed_wins"], X_train["windspeed_outlier"] = aplicar_winsorization(X_train, "windspeed", limites_windspeed)
X_test["windspeed_wins"], X_test["windspeed_outlier"] = aplicar_winsorization(X_test, "windspeed", limites_windspeed)

X_train["hum_wins"], X_train["hum_outlier"] = aplicar_winsorization(X_train, "hum", limites_hum)
X_test["hum_wins"], X_test["hum_outlier"] = aplicar_winsorization(X_test, "hum", limites_hum)

# Target Encoding
media_por_hora = y_train.groupby(X_train["hr"]).mean()
media_global = y_train.mean()

X_train["hr_target_enc"] = X_train["hr"].map(media_por_hora).fillna(media_global)
X_test["hr_target_enc"] = X_test["hr"].map(media_por_hora).fillna(media_global)

# Variables ciclicas
X_train["hr_sin"], X_train["hr_cos"] = transformar_ciclica(X_train["hr"], 24)
X_test["hr_sin"], X_test["hr_cos"] = transformar_ciclica(X_test["hr"], 24)

X_train["mnth_sin"], X_train["mnth_cos"] = transformar_ciclica(X_train["mnth"] - 1, 12)
X_test["mnth_sin"], X_test["mnth_cos"] = transformar_ciclica(X_test["mnth"] - 1, 12)

features_finales = [
    "season", "yr", "holiday", "weekday", "workingday", "weathersit",
    "temp", "atemp",
    "windspeed_wins", "windspeed_outlier",
    "hum_wins", "hum_outlier",
    "hr_target_enc",
    "hr_sin", "hr_cos",
    "mnth_sin", "mnth_cos"
]

X_train_fe = X_train[features_finales]
X_test_fe = X_test[features_finales]

X_train_fe.head()


,season,yr,holiday,weekday,workingday,weathersit,temp,atemp,windspeed_wins,windspeed_outlier,hum_wins,hum_outlier,hr_target_enc,hr_sin,hr_cos,mnth_sin,mnth_cos
17440,1,1,0,5,1,1,0.30,0.3030,0.1343,0,0.49,0,312.100520,-0.866025,-0.500000,-0.500000,0.866025
16867,4,1,0,2,1,1,0.48,0.4697,0.2239,0,0.67,0,309.410562,-0.965926,0.258819,-0.500000,0.866025
14851,3,1,0,1,1,1,0.62,0.6212,0.2985,0,0.38,0,309.410562,-0.965926,0.258819,-0.866025,-0.500000
5446,3,0,0,1,1,1,0.66,0.6212,0.2537,0,0.69,0,130.043706,-0.500000,0.866025,-0.500000,-0.866025
6025,3,0,0,5,1,2,0.62,0.5152,0.1642,0,0.93,1,32.613636,0.258819,0.965926,-0.866025,-0.500000


## Seccion A: Que es PyCaret y como funciona

**PyCaret** es una libreria de AutoML de "bajo codigo" (low-code): en vez de escribir el ciclo completo de entrenar, validar y comparar cada modelo uno por uno, PyCaret automatiza ese proceso y nos entrega un ranking de modelos con una funcion.

### Como funciona, paso a paso

1. **`setup()`**: es el primer paso siempre. Le pasamos el dataset y el nombre de la columna objetivo. Internamente, PyCaret arma un pipeline de preprocesamiento (maneja nulos, codifica variables categoricas, escala variables numericas, separa train/test, configura la validacion cruzada, etc.), todo de forma automatica.

2. **`compare_models()`**: esta es la funcion estrella. Entrena y evalua con validacion cruzada mas de 20 modelos distintos (Regresion Lineal, Ridge, Random Forest, XGBoost, LightGBM, KNN, SVM, entre otros), y devuelve una tabla ordenada por la metrica que elijamos (por ejemplo, RMSE), con el mejor modelo arriba de todo.

3. A partir de ahi, se puede seguir afinando el mejor modelo con funciones como `tune_model()` (ajusta hiperparametros), `finalize_model()` (entrena con el 100% de los datos) y `predict_model()` (genera predicciones).

### Como se ve el codigo en la practica

Este es el codigo tipico de un flujo de trabajo con PyCaret (no lo vamos a ejecutar en este notebook, ver el motivo mas abajo):

```python
from pycaret.regression import setup, compare_models, tune_model, finalize_model

# Paso 1: Configuracion. datos_pycaret debe incluir las features y la columna objetivo (session_id es como random seed).
exp = setup(data=datos_pycaret, target="cnt", session_id=42)

# Paso 2: Compara mas de 20 modelos con validacion cruzada y devuelve el mejor.
mejor_modelo = compare_models()

# Paso 3 (opcional): ajusta hiperparametros del mejor modelo encontrado.
modelo_afinado = tune_model(mejor_modelo)

# Paso 4 (opcional): entrena la version final con todos los datos disponibles.
modelo_final = finalize_model(modelo_afinado)
```

Con apenas estas lineas, `compare_models()` ya nos habria hecho el trabajo equivalente a entrenar y evaluar 20 modelos distintos a mano.

### Por que no lo ejecutamos en este notebook

PyCaret actualmente solo es compatible con **Python 3.9, 3.10 y 3.11**. Muchos entornos (incluyendo la version por defecto de Google Colab al momento de armar este material) ya corren en **Python 3.12**, donde PyCaret no se puede instalar. Pueden probar por su cuenta en un entorno con la version de Python correcta (por ejemplo, creando un entorno virtual local con Python 3.11).


## Seccion B: TPOT y la programacion genetica

**TPOT** (Tree-based Pipeline Optimization Tool) es otra libreria de AutoML, pero con una diferencia clave respecto a PyCaret: TPOT no solo elige el modelo, sino que **evoluciona el pipeline completo** (que pasos de preprocesamiento usar, en que orden, y con que modelo al final) usando **programacion genetica**.

### Como funciona la programacion genetica, a grandes rasgos

La idea esta inspirada en la evolucion biologica:

1. TPOT genera una **poblacion** inicial de pipelines al azar (cada pipeline es como un "individuo": una combinacion de pasos de preprocesamiento + un modelo).
2. Evalua que tan bien funciona cada pipeline (su "fitness"), usando validacion cruzada sobre los datos de entrenamiento.
3. Los pipelines con mejor desempenio tienen mas chances de "reproducirse": TPOT combina partes de dos pipelines buenos (**crossover**) o modifica levemente un pipeline (**mutacion**) para generar la siguiente **generacion**.
4. Este proceso se repite durante varias generaciones, y en teoria, la poblacion va mejorando con el tiempo, igual que en la evolucion biologica.

Esto es distinto a lo que hicimos con Optuna: Optuna optimizaba los hiperparametros de un modelo que nosotros ya habiamos elegido (Random Forest). TPOT va un paso mas alla, y tambien decide que pasos de preprocesamiento incluir y que modelo usar, no solo sus hiperparametros.

### Manos a la obra

Instalamos TPOT. A diferencia de PyCaret, esta version de TPOT si es compatible con Python 3.12, por lo que la podemos correr en el mismo entorno que usamos durante todo el taller.


In [ ]:
!pip install tpot --quiet


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 762.7 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.1/215.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [ ]:
# asegurarse que funciono (a pesar del mensaje de error)
import tpot
from tpot import TPOTRegressor

print(tpot.__version__)

1.1.0


Configuramos `TPOTRegressor` con una poblacion y un numero de generaciones chicos, para que la busqueda termine en unos pocos minutos. En un proyecto real, conviene subir estos valores (mas poblacion, mas generaciones) para que TPOT explore mas opciones, a costa de mas tiempo de computo.

Para mantener los tiempos razonables, tambien vamos a entrenar sobre una muestra del conjunto de entrenamiento en lugar de usarlo completo.


In [ ]:
# Selecciona una muestra del train set
X_train_muestra = X_train_fe.sample(1500, random_state=RANDOM_STATE)
# Obtiene los valores del target correspondientes
y_train_muestra = y_train.loc[X_train_muestra.index]

# Configura el modelo TPOT
tpot = TPOTRegressor(
    generations=2,              # Número de generaciones del algoritmo evolutivo
    population_size=6,          # Número de modelos candidatos evaluados en cada generación
    cv=3,                       # Validación cruzada de 3 folds para evaluar cada candidato
    random_state=RANDOM_STATE,  # Garantiza resultados reproducibles
    n_jobs=2,                   # Utiliza 2 núcleos del procesador
    verbose=2                   # Muestra información detallada del progreso
)

# Ejecuta la búsqueda automática del mejor pipeline de Machine Learning
tpot.fit(X_train_muestra, y_train_muestra)


/usr/local/lib/python3.12/dist-packages/tpot/tpot_estimator/estimator.py:458: UserWarning: Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.
  warnings.warn("Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.")
INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:43115
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:37397'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45215'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:44025 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:44025
I

TPOTRegressor(cv=3, n_jobs=2, random_state=42,
              search_space=<tpot.search_spaces.pipelines.sequential.SequentialPipeline object at 0x7d074b9f25a0>)

### Evaluando el mejor pipeline encontrado

Una vez que termina la busqueda, `tpot` ya se comporta como cualquier modelo de scikit-learn: podemos llamar a `.predict()` directamente sobre el conjunto de prueba.


In [ ]:
# Utiliza el mejor pipeline encontrado por TPOT para predecir sobre el conjunto de prueba
y_pred_tpot = tpot.predict(X_test_fe)

# Calcula las métricas de desempeño del modelo
rmse_tpot = np.sqrt(mean_squared_error(y_test, y_pred_tpot))
r2_tpot = r2_score(y_test, y_pred_tpot)

# Muestra el rendimiento final del pipeline seleccionado automáticamente por TPOT
print(f"RMSE con TPOT: {rmse_tpot:.2f}")
print(f"R2 con TPOT: {r2_tpot:.4f}")

RMSE con TPOT: 60.13
R2 con TPOT: 0.8842


Tambien podemos inspeccionar el pipeline que encontro, para ver que pasos de preprocesamiento y que modelo eligio.


In [ ]:
print(tpot.fitted_pipeline_)


Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                ('rfe',
                 RFE(estimator=ExtraTreesRegressor(criterion='absolute_error',
                                                   max_features=0.651217004502,
                                                   min_samples_leaf=14,
                                                   min_samples_split=18,
                                                   n_jobs=1, random_state=42),
                     step=0.9162776237062)),
                ('featureunion-1',
                 FeatureUnion(transformer_list=[('featureunion',
                                                 FeatureUnion(transformer_list=[('binarizer',
                                                                                 Binariz...
                              feature_types=None, feature_weights=None,
                              gamma=0.0153706668713, grow_policy=None,
                              importance_type=None,
                 

### Pipeline encontrado por TPOT

En lugar de buscar únicamente el mejor modelo, **TPOT optimiza un pipeline completo de Machine Learning**. En este caso, el pipeline seleccionado contiene varias etapas:

1. **`MinMaxScaler`**: normaliza las variables para que todas tengan una escala comparable.
2. **`RFE (Recursive Feature Elimination)`**: selecciona automáticamente las variables más relevantes utilizando un modelo `ExtraTreesRegressor`.
3. **`FeatureUnion`**: combina distintas transformaciones de las variables en un único conjunto de características.
4. **`XGBRegressor`**: utiliza un modelo de **Extreme Gradient Boosting (XGBoost)** como modelo final de regresión.

Este resultado ilustra una de las principales ventajas de TPOT: **no solo busca el mejor algoritmo**, sino también la mejor combinación de pasos de preprocesamiento, selección de variables y modelo final, optimizando automáticamente todo el flujo de trabajo.

### Una aclaracion sobre estos resultados

Con una poblacion y un numero de generaciones tan chicos (para que la demo entre en el tiempo de clase), es totalmente esperable que TPOT no supere al modelo que ya optimizamos a mano con Optuna en la Fase 3. La busqueda genetica necesita explorar muchas mas combinaciones (poblaciones mas grandes, mas generaciones, mas datos de entrenamiento) para encontrar pipelines realmente competitivos. La idea de esta seccion no es "ganarle" al modelo anterior, sino entender como funciona una herramienta de AutoML basada en programacion genetica.

Si quieren experimentar mas alla de la clase, pueden subir `generations` y `population_size`, o entrenar con `X_train_fe` completo en lugar de la muestra, y comparar cuanto mejora el resultado.


## Cierre

En este material opcional vimos dos enfoques distintos de AutoML:

- **PyCaret**: bajo codigo, compara muchos modelos con `compare_models()`, pero requiere Python 3.9-3.11.
- **TPOT**: usa programacion genetica para evolucionar pipelines completos (preprocesamiento + modelo), y es compatible con versiones mas recientes de Python.

Ambas herramientas apuntan al mismo objetivo que vimos con Optuna: reducir el trabajo manual de prueba y error al momento de encontrar un buen modelo. La diferencia esta en el alcance: Optuna optimiza los hiperparametros de un modelo que nosotros elegimos, mientras que PyCaret y TPOT tambien ayudan a elegir (o construir) el pipeline y el modelo en si.
